In [ ]:
# Manual Template Testing
SCORING_TEMPLATES = {
    ("Student", "strict"): {
        "surplus_ratio": (0.1, 10),
        "income_volatility": (0.5, 8),
        "gambling_ratio": (0.05, 12),
        "fixed_cost_ratio": (0.5, 5),
        "net_negative_months": (3, 10),
        "min_monthly_income": (0, 8),
        "max_monthly_expense": (1.5, 6),
        "emergency_buffer_ratio": (0.3, 12)
    },
    ("Student", "relaxed"): {
        "surplus_ratio": (0.05, 8),
        "income_volatility": (0.7, 5),
        "gambling_ratio": (0.08, 9),
        "fixed_cost_ratio": (0.6, 4),
        "net_negative_months": (4, 7),
        "min_monthly_income": (0, 5),
        "max_monthly_expense": (2.0, 5),
        "emergency_buffer_ratio": (0.25, 10)
    },
    ("Freelancer", "strict"): {
        "surplus_ratio": (0.2, 10),
        "income_volatility": (0.6, 12),
        "gambling_ratio": (0.10, 10),
        "fixed_cost_ratio": (0.4, 8),
        "net_negative_months": (3, 12),
        "min_monthly_income": (0, 10),
        "max_monthly_expense": (1.7, 7),
        "emergency_buffer_ratio": (0.4, 12)
    },
    ("Freelancer", "relaxed"): {
        "surplus_ratio": (0.10, 9),
        "income_volatility": (0.8, 7),
        "gambling_ratio": (0.15, 8),
        "fixed_cost_ratio": (0.6, 6),
        "net_negative_months": (4, 10),
        "min_monthly_income": (0, 7),
        "max_monthly_expense": (2.0, 7),
        "emergency_buffer_ratio": (0.3, 10)
    },
    ("Salaried", "strict"): {
        "surplus_ratio": (0.3, 12),
        "income_volatility": (0.3, 10),
        "gambling_ratio": (0.03, 15),
        "fixed_cost_ratio": (0.4, 5),
        "net_negative_months": (2, 10),
        "min_monthly_income": (0, 12),
        "max_monthly_expense": (1.3, 8),
        "emergency_buffer_ratio": (0.4, 15)
    },
    ("Salaried", "relaxed"): {
        "surplus_ratio": (0.2, 10),
        "income_volatility": (0.5, 7),
        "gambling_ratio": (0.06, 10),
        "fixed_cost_ratio": (0.5, 5),
        "net_negative_months": (3, 8),
        "min_monthly_income": (0, 8),
        "max_monthly_expense": (1.6, 7),
        "emergency_buffer_ratio": (0.3, 12)
    },
    ("Other", "strict"): {
        "surplus_ratio": (0.15, 10),
        "income_volatility": (0.7, 8),
        "gambling_ratio": (0.10, 8),
        "fixed_cost_ratio": (0.5, 5),
        "net_negative_months": (3, 10),
        "min_monthly_income": (0, 8),
        "max_monthly_expense": (1.5, 7),
        "emergency_buffer_ratio": (0.3, 12)
    },
    ("Other", "relaxed"): {
        "surplus_ratio": (0.05, 10),
        "income_volatility": (0.7, 7),
        "gambling_ratio": (0.10, 7),
        "fixed_cost_ratio": (0.5, 5),
        "net_negative_months": (4, 8),
        "min_monthly_income": (0, 5),
        "max_monthly_expense": (2.0, 5),
        "emergency_buffer_ratio": (0.25, 10)
    }
}

def assign_score_tier(score):
    if score >= 80:
        return "Green - Very Low Risk"
    elif score >= 60:
        return "Yellow - Medium Risk"
    elif score >= 40:
        return "Orange - High Risk"
    else:
        return "Red - Very High Risk"

def score_user_with_breakdown(user_features, template):
    score = 100
    breakdown = []

    for feature, (threshold, penalty) in template.items():
        value = user_features.get(feature, np.nan)
        if any(key in feature for key in ["volatility", "gambling", "fixed_cost", "net_negative_months", "max_monthly_expense"]):
            if value > threshold:
                score -= penalty
                contribution = -penalty
            else:
                contribution = 0
            expected_behavior = f"≤ {threshold}"
        else:
            if value < threshold:
                score -= penalty
                contribution = -penalty
            else:
                contribution = 0
            expected_behavior = f"≥ {threshold}"

        breakdown.append({
            "feature": feature,
            "value": value,
            "threshold": threshold,
            "expected_behavior": expected_behavior,
            "penalty_points": penalty,
            "final_contribution": contribution
        })

    return max(0, min(100, score)), breakdown

def full_scoring_with_explanations(df_features, desired_profile="Other", rental_context="relaxed"):
    template = SCORING_TEMPLATES.get((desired_profile, rental_context), SCORING_TEMPLATES[("Other", "relaxed")])
    all_breakdowns, scores, tiers = [], [], []

    for _, row in df_features.iterrows():
        final_score, breakdown = score_user_with_breakdown(row, template)
        scores.append(final_score)
        tiers.append(assign_score_tier(final_score))
        for b in breakdown:
            b["userId"] = row["userId"]
        all_breakdowns.extend(breakdown)

    df_features["financial_behavior_score"] = scores
    df_features["risk_tier"] = tiers
    df_breakdown = pd.DataFrame(all_breakdowns)

    return df_features, df_breakdown


In [ ]:
def evaluate_scoring_templates(df_scored):
    total_defaults = df_scored['defaulted'].sum()
    total_non_defaults = len(df_scored) - total_defaults

    high_risk_mask = df_scored['risk_tier'].isin(['Red - Very High Risk', 'Orange - High Risk'])

    defaults_in_high_risk = df_scored[high_risk_mask & (df_scored['defaulted'] == 1)].shape[0]
    non_defaults_in_high_risk = df_scored[high_risk_mask & (df_scored['defaulted'] == 0)].shape[0]

    risk_concentration = (defaults_in_high_risk / total_defaults) * 100 if total_defaults > 0 else 0
    false_positive_rate = (non_defaults_in_high_risk / total_non_defaults) * 100 if total_non_defaults > 0 else 0

    print(f"Risk Concentration: {risk_concentration:.2f}% of defaults are in Red and Orange tiers.")
    print(f"False Positive Rate: {false_positive_rate:.2f}% of non-defaults are incorrectly flagged in Red and Orange tiers.")

    return {
        'risk_concentration_percent': risk_concentration,
        'false_positive_rate_percent': false_positive_rate
    }


df_features_scored, _ = full_scoring_with_explanations(df_features, desired_profile="Other", rental_context="relaxed")
metrics = evaluate_scoring_templates(df_features_scored)
